In [13]:
from crewai import Agent,Task,Crew,Process,LLM
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing import List
from crewai.tools import tool
from scrapegraph_py import Client
import agentops
import os 
from typing import Optional

In [14]:
load_dotenv()

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2,
    max_tokens=300
)

In [15]:
session = agentops.init(
    api_key=os.getenv("AGENTOPS_API_KEY"),
    #this code to prevent close after first agent 
    skip_auto_end_session=True
)
print(session)

None


In [16]:
search_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
scrape_client = Client(api_key=os.getenv("ScrapGraphAI_API_KEY"))

C:\Users\PC\AppData\Local\Temp\ipykernel_25352\949116198.py:2: DeprecationWarning: scrapegraph-py v1.x is deprecated and will be removed in a future release. Please upgrade to scrapegraph-py v2.x for the new API surface. See migration guide: https://docs.scrapegraphai.com/transition-from-v1-to-v2
  scrape_client = Client(api_key=os.getenv("ScrapGraphAI_API_KEY"))


In [17]:
outpur_dir = "./ai-agent-output"
os.makedirs(outpur_dir,exist_ok=True)

### FIRST AGENT & TASK

In [18]:
no_keywords = 10
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine")

#every agent can make one task or multiple tasks
#each agent has role, goal, backstory,llm  
Search_Query_Recommendation_Agent = Agent(
    role="Search Query Strategist",
    goal="/n".join([
        "to provide alist of suggested search queries to be passed to the search engine",
        "the queries must be varied and looking for specific items"
           ]),
    backstory="You are an expert search query strategist specializing in product research and online shopping. You analyze user requests and transform them into clear, specific, and diverse search queries. Your goal is to cover different brands, models, specifications, price ranges, and purchasing options to help search engines find the most relevant products and deals."    ,
    llm = llm,
    verbose=True
)
#each task has description , expected output and agent to do this task , expected output , output json
# optional can take async,output file 
Search_Query_Recommendation_Task = Task(
    description="\n".join([
        "Generate exactly {no_keyword} highly relevant and diverse search keywords for finding {product_name}.",

        "The keywords will be passed to a separate search engine agent that will perform the actual web search.",

        "The product must be available for purchase and deliverable in {country_name}.",

        "The search will be restricted to these e-commerce websites:",
        "{website_list}",

        "IMPORTANT: Do NOT include website names, domains, URLs, or 'site:' operators in the keywords.",

        "Do NOT generate URLs or links.",

        "Do NOT return product pages or search results.",

        "Do NOT include blog posts, articles, reviews, forums, news websites, or informational content.",

        "Generate keywords that describe the product and its purchasing requirements.",

        "Vary the keywords using different brands, models, specifications, features, price ranges, and product variations.",

        "Each keyword should be a concise search phrase that can be directly passed to a search engine.",

        "Return ONLY the list of keywords.",
    ]),
    #json to prevent words in introduction and conclusion
    expected_output="A JSON containing alist of suggested search queries",
    #we will create pydantic scheme for output json 
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(outpur_dir,"step1.json"),
    agent=Search_Query_Recommendation_Agent
)

### SECOND AGENT

In [19]:
#create nested pydantic
class SingleSearchResult(BaseModel):
    title: str
    url: str
    content: str
    score: float
    search_query: str
    
class AllSearchResult(BaseModel):
    result:List[SingleSearchResult]


#  we need google search tool
#will make custom tool (tavily)
# doc string is must in custom tool to tell what tool make
@tool
def search_engine_tool(query:str):
    """
    Search the web for relevant e-commerce product pages using the provided query.

    Args:
        query: A search query describing the product or information to find.
                The query should be specific and focused on purchasable products.

    Returns:
        Search results containing relevant web pages matching the query.
    """
    return  search_client.search(
    query=query,
    search_depth="advanced",
    max_results=5
)


# take keywords and search google for products based on the suggested search query
Search_Engine_Agent = Agent(
    role = "Search Engine Agent",
    goal = "To search for products based on the suggessted search query",
    backstory="""
    You are an expert web search specialist focused on finding
    purchasable products across e-commerce websites.

    You receive carefully generated search queries from a Search Query
    Recommendation Agent and use them to perform accurate web searches.

    Your responsibility is to find relevant product pages, product listings,
    and online stores. You prioritize results that contain actual products
    available for purchase and ignore blogs, news articles, forums, and
    informational pages.

    You carefully follow the provided search queries and return the most
    relevant search results without modifying the user's search intent.
    """,
    llm = llm,
    verbose=True,
    tools=[search_engine_tool]

)

Search_Engine_Task = Task(
    description="/n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output="A JSON object containing the search results",
    output_json=AllSearchResult,
    output_file=os.path.join(outpur_dir,"step2.json"),
    context=[Search_Query_Recommendation_Task],
    agent=Search_Engine_Agent
)

### Third Agent

In [20]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: str = Field(..., title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: Optional[float]  = Field(..., title="The current price of the product")
    product_original_price:  Optional[float] = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage:  Optional[float] = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

    product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)

    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]


#we need custom tool to descripe , extract data from webpages (ScrapGraphAi)
@tool
def web_scraping_tool(page_url: str):
    # doc string is must in custom tool to tell what tool make
    """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    """
    details = scrape_client.smartscraper(
        website_url=page_url,
        user_prompt="Extract ```json\n" + SingleExtractedProduct.schema_json()+ "```\nFrom the web page"
    )
    return {
        "page_url":page_url,
        "details":details
    }


# take web result and download page  to extract details from any web page
Scraping_Agent = Agent(
    role = "web scraping agent",
    goal = "to extract details from any website",
    backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm =llm ,
    tools = [web_scraping_tool],
    verbose = True
)
Scraping_Task = Task(
    description="\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Collect the best {top_recommendations_no} products from the search results.",       
    ]),
    expected_output="A JSON object containing products details",
    output_json=AllExtractedProducts,
    output_file=(os.path.join(outpur_dir,"step3.json")),
    agent=Scraping_Agent
)

C:\Users\PC\AppData\Local\Temp\ipykernel_25352\2032913067.py:14: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)
C:\Users\PC\AppData\Local\Temp\ipykernel_25352\2032913067.py:14: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)


### Fourth Agent

In [21]:
#last agent take data from third agent and generate  aprofessional procurement report
# we need to HTML generator Tool it native skill we dont need a tool
procurement_report_author_agent = Agent(
    role="procurement report author agent",
    goal="To generate a professional HTML page for the procurement report",
    backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm=llm,
    verbose=True,
)
procurement_report_author_Task = Task(
  description="\n".join([
        "The task is to generate a professional HTML page for the procurement report.",
        # to make visual more stander and readable ==> bootstrap css
        "You have to use Bootstrap CSS framework for a better UI.",
        "Use the provided context about the company to make a specialized report.",
        "The report will include the search results and prices of products from different websites.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Any additional information, such as raw data or supplementary materials.",
    ]),

    expected_output="A professional HTML page for the procurement report.",
    output_file=os.path.join(outpur_dir, "step_4_procurement_report.html"),
    agent=procurement_report_author_agent,
)

In [22]:
run = Crew(
    agents=[
        Search_Query_Recommendation_Agent,
        Search_Engine_Agent, 
        Scraping_Agent,
        procurement_report_author_agent
            ],
    tasks=[
        Search_Query_Recommendation_Task,
        Search_Engine_Task,
        Scraping_Task,
        procurement_report_author_Task
        ],
    process=Process.sequential
)

In [23]:
results =  await run.kickoff_async(
    inputs={
        "product_name": "coffee machine for the office",
        "website_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keyword": 10,
        "score_th":0.8,
        "top_recommendations_no":5,
        
        
    }
    )

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Query Strategist                                                                                 │
│                                                                                                                 │
│  Task: Generate exactly 10 highly relevant and diverse search keywords for finding coffee machine for the       │
│  office.                                                                                                        │
│  The keywords will be passed to a separate search engine agent that will perform the actual web search.         │
│  The product must be available for purchase and deliverable in Egypt.                                           │
│  The search will be restricted to these e-commerce websites:                                                    │
│  ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']                                                 │
│  IMPORTANT: Do NOT include website names, domains, URLs, or 'site:' operators in the keywords.                  │
│  Do NOT generate URLs or links.                                                                                 │
│  Do NOT return product pages or search results.                                                                 │
│  Do NOT include blog posts, articles, reviews, forums, news websites, or informational content.                 │
│  Generate keywords that describe the product and its purchasing requirements.                                   │
│  Vary the keywords using different brands, models, specifications, features, price ranges, and product          │
│  variations.                                                                                                    │
│  Each keyword should be a concise search phrase that can be directly passed to a search engine.                 │
│  Return ONLY the list of keywords.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Query Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  queries=['commercial office coffee machine espresso maker egypt', 'bean to cup commercial coffee machine for   │
│  office', 'delonghi automatic coffee machine for office use', 'nespresso commercial capsule coffee machine      │
│  egypt', 'philips fully automatic espresso machine office price', 'large capacity filter coffee maker for       │
│  office', 'commercial cappuccino machine for office workplace', 'dual boiler espresso coffee machine office     │
│  delivery', 'affordable office coffee maker 220v egypt', 'professional espresso machine with grinder for        │
│  office']                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Task: The task is to search for products based on the suggested search queries./nYou have to collect results   │
│  from multiple search queries./nIgnore any susbicious links or not an ecommerce single product website          │
│  link./nIgnore any search results with confidence score less than (0.8) ./nThe search results will be used to   │
│  compare prices of products from different websites.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_engine_tool executed with result: {'query': 'commercial office coffee machine espresso maker egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-espresso-nespresso-coffee-m...
[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: web scraping agent                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "products": [                                                                                                │
│      {                                                                                                          │
│        "page_url": "https://www.fajtradingllc.com/pages/coffee-machines-in-egypt",                              │
│        "product_title": "Jetinno JL32 Espresso Coffee Machine",                                                 │
│        "product_image_url": "https://www.fajtradingllc.com/cdn/shop/files/jetinno-jl32.jpg",                    │
│        "product_url": "https://www.fajtradingllc.com/pages/coffee-machines-in-egypt",                           │
│        "product_current_price": 45000.0,                                                                        │
│        "product_original_price": 50000.0,                                                                       │
│        "product_discount_percentage": 10.0,                                                                     │
│        "product_specs": [                                                                                       │
│          {                                                                                                      │
│            "specification_name": "Type",                                                                        │
│            "specification_value": "Bean-to-Cup Commercial Espresso Machine"                                     │
│          },                                                                                                     │
│          {                                                                                                      │
│            "specification_name": "Application",                                                                 │
│            "specification_value": "Office & Commercial Space"                                                   │
│          }                                                                                                      │
│        ],                                                                                                       │
│        "agent_recommendation_rank": 5,                                                                          │
│        "agent_recommendation_notes": [                                                                          │
│          "Top-tier bean-to-cup machine designed specifically to boost productivity with cafe-quality office     │
│  coffee.",                                                                                                      │
│          "Commercial-grade performance suitable for workplace environments."                                    │
│        ]                                                                                                        │
│      },                                                                                                         │
│      {                                                                                                          │
│        "page_url": "https://www.fajtradingllc.com/pages/coffee-machines-in-egypt",                              │
│        "product_title": "De'Longhi Dinamica ECAM350.55.

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: procurement report author agent                                                                         │
│                                                                                                                 │
│  Task: The task is to generate a professional HTML page for the procurement report.                             │
│  You have to use Bootstrap CSS framework for a better UI.                                                       │
│  Use the provided context about the company to make a specialized report.                                       │
│  The report will include the search results and prices of products from different websites.                     │
│  The report should be structured with the following sections:                                                   │
│  1. Executive Summary: A brief overview of the procurement process and key findings.                            │
│  2. Introduction: An introduction to the purpose and scope of the report.                                       │
│  3. Methodology: A description of the methods used to gather and compare prices.                                │
│  4. Findings: Detailed comparison of prices from different websites, including tables and charts.               │
│  5. Analysis: An analysis of the findings, highlighting any significant trends or observations.                 │
│  6. Recommendations: Suggestions for procurement based on the analysis.                                         │
│  7. Conclusion: A summary of the report and final thoughts.                                                     │
│  8. Appendices: Any additional information, such as raw data or supplementary materials.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: procurement report author agent                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```html                                                                                                        │
│  <!DOCTYPE html>                                                                                                │
│  <html lang="en">                                                                                               │
│  <head>                                                                                                         │
│      <meta charset="UTF-8">                                                                                     │
│      <meta name="viewport" content="width=device-width, initial-scale=1.0">                                     │
│      <title>Commercial Office Coffee Machine Procurement Report - Egypt Market 2025</title>                     │
│      <!-- Bootstrap CSS CDN -->                                                                                 │
│      <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/css/bootstrap.min.css" rel="stylesheet">          │
│      <!-- FontAwesome for Icons -->                                                                             │
│      <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.2/css/all.min.css">   │
│      <style>                                                                                                    │
│          body {                                                                                                 │
│              background-color: #f8f9fa;                                                                         │
│              color: #333;                                                                                       │
│              font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;                                      │
│          }                                                                                                      │
│          .hero-section {                                                                                        │
│              background: linear-gradient(135deg, #2c3e50, #4ca1af);                                             │
│              color: white;                                                                                      │
│              padding: 4rem 2rem;                                                                                │
│              margin-bottom: 2rem;                                                                               │
│              border-radius: 0 0 1rem 1rem;                                                                      │
│          }                                                                                                      │
│          .card {                                                                                                │
│              border: none;                                                                                      │
│              box-shadow: 0 4px 6px rgba(0,0,0,0.05);                                                            │
│              transition: transform 0.2s;                                                                        │
│              margin-bottom: 1.5rem;                                                                             │
│          }                                             

In [24]:
results

CrewOutput(raw='```html\n<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>Commercial Office Coffee Machine Procurement Report - Egypt Market 2025</title>\n    <!-- Bootstrap CSS CDN -->\n    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/css/bootstrap.min.css" rel="stylesheet">\n    <!-- FontAwesome for Icons -->\n    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.2/css/all.min.css">\n    <style>\n        body {\n            background-color: #f8f9fa;\n            color: #333;\n            font-family: \'Segoe UI\', Tahoma, Geneva, Verdana, sans-serif;\n        }\n        .hero-section {\n            background: linear-gradient(135deg, #2c3e50, #4ca1af);\n            color: white;\n            padding: 4rem 2rem;\n            margin-bottom: 2rem;\n            border-radius: 0 0 1rem 1rem;\n        }\n        .card {\n            bo